# FEATURE ENGINEERING

In [20]:
def clean_text(text):
    # ======= converting all the text to lower case ==========
    
    text = str(text).lower()

    # ========= keep words that start with only letters and numbers =================
    
    text = re.sub(r'[^0-9a-z\s]' , ' ' , text) 

    # ======= removing extra spaces(whitespaces) ===========
    
    text = re.sub(r'/s+' , ' ' , text)

    # ======== strip and return text =========
    text = text.strip()
    return text


def combine_text(row):

    # ========= Joining all 5 option with prompt in a single string =========
    
    combined = str(row['prompt']) + ' ' + str(row['A']) + ' ' + str(row['B']) + ' ' + str(row['C']) + ' ' + str(row['D']) + ' ' + str(row['E'])
    
    return combined

# ========= Apply to both train and test ==========

train['full_text'] = train.apply(combine_text , axis = 1)
test['full_text'] = test.apply(combine_text , axis = 1)

y_train = train['answer']

print('Preprocessing Completed')
print('='*50)
print('Sample combined text :')
print()
print(train['full_text'].iloc[0][:256])
    

Preprocessing Completed
Sample combined text :

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined 


In [23]:
# mAP@3 metric implementation
# this is the official evaluation metric for this competition
# if correct answer is at rank 1 we get score 1.0
# if correct answer is at rank 2 we get score 0.5
# if correct answer is at rank 3 we get score 0.33
# otherwise we get 0

def average_precision_at_k(actual, predicted, k=3):
    predicted = predicted[:k]
    score = 0.0
    num_hits = 0
    for i in range(len(predicted)):
        if predicted[i] == actual:
            num_hits += 1
            score += num_hits / (i + 1.0)
    return score


def map_at_k(actuals, predictions, k=3):
    total = 0.0
    for i in range(len(actuals)):
        total += average_precision_at_k(actuals[i], predictions[i], k)
    return total / len(actuals)


def get_top3_from_probs(probs, classes):
    # ======= sorting probabilities in descending order and returning top 3 class labels =========
    sorted_indices = np.argsort(probs)[::-1]
    top3 = [classes[idx] for idx in sorted_indices[:3]]
    return top3


# ======== quick test to make sure mAP@3 is working correctly ============
test_actual  = ['B', 'A', 'C']
test_preds   = [['B', 'A', 'C'], ['C', 'A', 'B'], ['C', 'A', 'B']]
result = map_at_k(test_actual, test_preds)
print('Sanity check : ')
print('='*50)
print()
print('mAP@3 test result:', round(result, 4))
print()
print('expected: 0.8333')

Sanity check : 

mAP@3 test result: 0.8333

expected: 0.8333
